# Práctica 2 · Actividad 2: Data Profiling de Titanic

En este notebook hacemos el *perfilado de datos* (data profiling) del dataset del **Titanic**: 891 pasajeros con su clase, sexo, edad, familiares a bordo, tarifa, cabina y puerto de embarque, y una columna `Survived` que indica si sobrevivieron. La idea es conocer bien los datos antes de pensar en modelos: qué tipo de variables hay, si falta información, si hay valores raros y si algo podría confundir a un modelo.

Seguimos la tabla de la Actividad 2. Para las variables **numéricas** y **categóricas** revisamos el tipo de dato, los ausentes, los valores únicos, los estadísticos, la moda y el dominio. Además buscamos filas duplicadas, columnas constantes, categorías inconsistentes, identificadores, correlaciones altas y posibles fugas de información. Debajo de cada tabla explicamos con palabras sencillas qué significan los números, y al final juntamos los hallazgos más importantes para el producto parcial.

**Cómo usarlo:** ejecuta las celdas de arriba hacia abajo (Shift + Enter).

> **Declaración de uso de IA generativa:** este código se generó con asistencia de Claude (Anthropic). Ayudó a mejorar el estilo visual de las tablas y, en lugar de entregar solo un script, sugirió usar un notebook para mostrar mejor la información solicitada.

## 0. Instalación (solo la primera vez)
Si ya tienes `pandas`, `numpy` y `jinja2` instalados, puedes saltarte esta celda. `jinja2` es lo que permite darles formato a las tablas.

In [ ]:
%pip install pandas numpy jinja2

## 1. Librerías, carga de datos y configuración
Aquí importamos las librerías, cargamos el CSV y definimos cómo vamos a tratar cada columna. `Survived` es la variable objetivo (1 = sobrevivió, 0 = no). Marcamos `PassengerId` y `Name` como identificadores porque son únicos en cada fila. `Pclass` es un número (1, 2 o 3), pero representa la clase del boleto, así que la tratamos como categórica ordinal. `SibSp` y `Parch` son numéricas discretas (tienen pocos valores distintos), por eso también les calculamos frecuencias.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# --- Carga de datos (carpeta datasets/ junto a este notebook) ---
df = pd.read_csv("datasets/Titanic-Dataset.csv")

# --- Configuración del dataset ---
OBJETIVO = "Survived"
POSITIVA = 1                               # valor de la clase que consideramos "positiva"
IDENTIFICADORES = ["PassengerId", "Name"]  # únicos por fila
NUMERICAS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAS = ["Pclass", "Sex", "Ticket", "Cabin", "Embarked"]   # Pclass es ordinal (1, 2, 3)
FRECUENCIAS = ["Pclass", "Sex", "Embarked", "Cabin", "Ticket"]   # columnas cuyas frecuencias queremos ver
DISCRETAS = ["SibSp", "Parch"]             # numéricas con pocos valores (frecuencias opcionales)
COLS_CORR = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
UMBRAL_CORR = 0.7

df.head()

### Estilo de las tablas
Esta celda solo define el formato de las tablas (líneas finas y un par de colores suaves para resaltar lo que conviene revisar). No hace ningún análisis, así que no hay nada que interpretar.

In [ ]:
# --- Estilo común de las tablas (simple y limpio) ---
GRIS = "rgba(128,128,128,0.45)"      # líneas del encabezado
SUAVE = "rgba(128,128,128,0.18)"     # líneas entre filas
AMARILLO = "background-color:#FFF3CD;color:#000"   # para revisar
ROJO = "background-color:#F8D7DA;color:#000"       # ausencia alta

ESTILO = [
    {"selector": "table", "props": [("border-collapse", "collapse"), ("font-size", "14px")]},
    {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"),
                                       ("font-size", "15px"), ("font-weight", "bold"), ("padding", "6px 0")]},
    {"selector": "th", "props": [("padding", "6px 12px"), ("text-align", "left")]},
    {"selector": "td", "props": [("padding", "6px 12px"), ("border-bottom", f"1px solid {SUAVE}"),
                                 ("max-width", "560px"), ("white-space", "normal")]},
    {"selector": "th.col_heading", "props": [("border-bottom", f"2px solid {GRIS}")]},
    {"selector": "th.blank", "props": [("border-bottom", f"2px solid {GRIS}")]},
    {"selector": "th.row_heading", "props": [("border-bottom", f"1px solid {SUAVE}"), ("font-weight", "600")]},
]

def bonito(df, titulo):
    """Aplica el estilo común y un título a una tabla."""
    d = df.copy()
    d.index.name = None          # evita una fila extra con el nombre del índice
    return d.style.set_caption(titulo).set_table_styles(ESTILO)

def aplicar(styler, funcion, subset):
    """Aplica una función de color celda por celda (compatible con pandas viejos y nuevos)."""
    if hasattr(styler, "map"):
        return styler.map(funcion, subset=subset)
    return styler.applymap(funcion, subset=subset)

def color_ausencia(v):
    """Sin color: sin ausentes. Amarillo: menos de 30%. Rojo: 30% o más."""
    if v == 0:
        return ""
    return AMARILLO if v < 30 else ROJO

def color_alerta(v):
    """Amarillo para 'Sí' o para cualquier número mayor que cero."""
    if v == "Sí" or (isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) and v > 0):
        return AMARILLO
    return ""

def titulo(texto):
    display(Markdown(texto))


## 2. Análisis solicitado en la práctica

### 2.1 Tipo de dato, valores válidos y ausentes
*(numéricas: sí · categóricas: sí)*

Empezamos por ver cómo reconoció pandas cada columna: qué tipo de dato le asignó, cuántos valores válidos (no nulos) tiene y cuántos le faltan.

In [ ]:
def tipo_valores_ausentes(df):
    """Tipo de dato, clase de variable, valores no nulos (válidos) y ausentes por columna."""
    def clase(c):
        if c == OBJETIVO: return "Objetivo"
        if c in IDENTIFICADORES: return "Identificador"
        if c in NUMERICAS: return "Numérica"
        return "Categórica"
    t = pd.DataFrame({
        "Tipo de dato": df.dtypes.astype(str),
        "Clase": [clase(c) for c in df.columns],
        "Valores no nulos": df.notna().sum(),
        "Ausentes": df.isna().sum(),
    })
    return bonito(t, "Tipo de dato y valores ausentes").format(
        {"Valores no nulos": "{:,}", "Ausentes": "{:,}"})

tipo_valores_ausentes(df)

Faltan datos en tres columnas: `Cabin` (687 ausentes, con solo 204 valores válidos), `Age` (177 ausentes) y `Embarked` (2). El resto viene completo.

`Age` es decimal (`float64`) porque hay edades fraccionarias, como las de los bebés. Y otra vez el tipo de dato no cuenta toda la historia: `Pclass`, `SibSp`, `Parch` y `Survived` son enteros, pero `Pclass` y `Survived` funcionan en realidad como categorías.

### 2.2 Porcentaje de ausencia y valores únicos
*(numéricas: sí · categóricas: sí)*

Aquí medimos qué porcentaje de cada columna está vacío. Las celdas con ausentes se resaltan: amarillo si falta menos del 30% y rojo si falta el 30% o más. También contamos los valores únicos, que sirven para distinguir categóricas de continuas y para detectar identificadores (una columna con casi todos los valores distintos suele ser uno).

In [ ]:
def ausencia_y_unicos(df):
    t = pd.DataFrame({
        "% de ausencia": (df.isna().mean() * 100).round(2),
        "Valores únicos": df.nunique(),
        "% de únicos": (df.nunique() / len(df) * 100).round(1),
    })
    est = bonito(t, "Ausencia y valores únicos")
    est = est.format({"% de ausencia": "{:.2f}%", "Valores únicos": "{:,}", "% de únicos": "{:.1f}%"})
    return aplicar(est, color_ausencia, ["% de ausencia"])

ausencia_y_unicos(df)

La ausencia más grave es la de `Cabin`: 77.10%, resaltada en rojo. `Age` falta en casi 1 de cada 5 pasajeros (19.87%), demasiado como para ignorarlo. `Embarked` solo pierde 2 filas (0.22%), casi nada.

En valores únicos, `PassengerId` y `Name` tienen 100%, así que son identificadores. `Ticket` llega a 76.4%: no es único por fila porque varias personas comparten boleto, un detalle al que volvemos en la sección de fugas. Por último, `Cabin` tiene 147 valores distintos entre sus 204 datos válidos.

### 2.3 Mínimo, máximo, media, mediana y desviación estándar
*(solo numéricas)*

Un resumen rápido de la escala y el centro de cada variable numérica. Comparar la media con la mediana da una primera pista de cómo se reparten los datos.

In [ ]:
def resumen_numericas(df, cols):
    t = pd.DataFrame({
        "Mínimo": df[cols].min(),
        "Máximo": df[cols].max(),
        "Media": df[cols].mean(),
        "Mediana": df[cols].median(),
        "Desviación estándar": df[cols].std(),
    })
    return bonito(t, "Resumen de variables numéricas").format("{:,.2f}")

resumen_numericas(df, NUMERICAS)

`Age` se comporta bien: va de 0.42 a 80 años, con media (29.7) y mediana (28) muy cercanas. `Fare` es todo lo contrario: su media (32.20) es más del doble de su mediana (14.45), su desviación estándar (49.69) supera a la media y el máximo llega a 512.33. En pocas palabras, la mayoría de los pasajeros pagó tarifas bajas y un grupo pequeño pagó muchísimo más. `SibSp` y `Parch` tienen mediana 0, es decir que la mayor parte de los pasajeros viajaba sin familiares a bordo.

### 2.4 Percentiles y asimetría
*(solo numéricas)*

Los percentiles muestran cómo se reparten los datos (por ejemplo, el P25 es el valor por debajo del cual está el 25% de los clientes). La asimetría resume la forma de la distribución: cerca de 0 es simétrica, y mayor a 1 en valor absoluto indica una cola larga con probables valores extremos. Si la media queda muy por encima de la mediana, la asimetría es positiva (cola hacia la derecha).

In [ ]:
def percentiles_asimetria(df, cols):
    t = df[cols].quantile([.01, .05, .25, .5, .75, .95, .99]).T
    t.columns = ["P1", "P5", "P25", "P50", "P75", "P95", "P99"]
    t["Asimetría"] = df[cols].skew()
    def interpretar(a):
        fuerza = "simétrica" if abs(a) < 0.5 else ("asimetría moderada" if abs(a) < 1 else "asimetría fuerte")
        if fuerza == "simétrica":
            return fuerza
        return fuerza + (" a la derecha" if a > 0 else " a la izquierda")
    t["Interpretación"] = t["Asimetría"].apply(interpretar)
    est = bonito(t, "Percentiles y asimetría").format("{:,.2f}", subset=t.columns[:-1])
    def color_asim(v):
        return AMARILLO if abs(v) > 1 else ""
    return aplicar(est, color_asim, ["Asimetría"])

percentiles_asimetria(df, NUMERICAS)

`Age` es simétrica (0.39). En cambio, `Fare` (4.79), `SibSp` (3.70) y `Parch` (2.75) tienen asimetría fuerte hacia la derecha y quedan resaltadas en la tabla.

En `Fare` se nota en los percentiles: el 75% de los pasajeros pagó 31 o menos, pero el P99 llega a 249. En `SibSp` el P75 es 1 y en `Parch` es 0, así que los valores altos corresponden a familias numerosas: son raras, pero reales, no errores. Más adelante habrá que decidir cómo tratar esas colas, por ejemplo con una transformación logarítmica en `Fare` o agrupando los valores altos de `SibSp` y `Parch`.

### 2.5 Moda y frecuencias por categoría
*(categóricas: sí · numéricas: opcional)*

Para las categóricas primero identificamos la categoría más común (la moda) y qué tan dominante es. Después mostramos las frecuencias completas de algunas columnas, y también las de las numéricas discretas (`SibSp` y `Parch`), que es la parte opcional de la tabla.

In [ ]:
def resumen_moda(df, cols):
    filas = []
    for c in cols:
        vc = df[c].value_counts()
        filas.append({"Columna": c, "Categorías": df[c].nunique(), "Moda": vc.index[0],
                      "Frecuencia de la moda": int(vc.iloc[0]),
                      "% de la moda": round(vc.iloc[0] / len(df) * 100, 2)})
    t = pd.DataFrame(filas).set_index("Columna")
    est = bonito(t, "Moda de las variables categóricas")
    est = est.format({"Categorías": "{:,}", "Frecuencia de la moda": "{:,}", "% de la moda": "{:.2f}%"})
    return est

def frecuencias(df, col, top=10):
    """Frecuencia y porcentaje por categoría (incluye ausentes). Muestra las 'top' más frecuentes."""
    vc = df[col].value_counts(dropna=False)
    t = pd.DataFrame({"Categoría": vc.index.astype(str), "Frecuencia": vc.values,
                      "%": (vc.values / len(df) * 100).round(2)}).head(top).set_index("Categoría")
    sufijo = f" (top {top} de {len(vc)})" if len(vc) > top else ""
    est = bonito(t, f"Frecuencias de {col}{sufijo}")
    est = est.format({"Frecuencia": "{:,}", "%": "{:.2f}%"})
    return est

display(resumen_moda(df, CATEGORICAS))
for c in FRECUENCIAS:
    display(frecuencias(df, c))

In [ ]:
# Opcional: frecuencias de numéricas discretas (pocos valores distintos)
for c in DISCRETAS:
    display(frecuencias(df, c))

Más de la mitad de los pasajeros (55.11%) viajaba en tercera clase y casi dos tercios (64.76%) eran hombres. La mayoría embarcó en el puerto `S` (72.28%), y 2 pasajeros no tienen puerto registrado.

En `Ticket` y `Cabin` la moda dice poco: tienen tantas categorías (681 y 147) que la más frecuente apenas llega a 0.79% y 0.45%. De hecho, en las frecuencias de `Cabin` lo más común es no tener dato (687 filas). Las frecuencias de `SibSp` y `Parch` confirman lo de las medianas: el 68.24% viajaba sin hermanos ni cónyuge, y el 76.09% sin padres ni hijos.

### 2.6 Dominio y valores potencialmente inválidos
*(numéricas: sí · categóricas: sí)*

Para cada columna definimos qué valores tienen sentido (su **dominio**) y contamos cuántos quedan fuera. Después listamos otros valores que no son inválidos, pero que conviene revisar.

In [ ]:
DOMINIO = [
    ("Survived", "0 o 1", lambda s: s.notna() & ~s.isin([0, 1])),
    ("Pclass", "1, 2 o 3", lambda s: s.notna() & ~s.isin([1, 2, 3])),
    ("Sex", "male o female", lambda s: s.notna() & ~s.isin(["male", "female"])),
    ("Age", "mayor que 0 y hasta 120 años", lambda s: s.notna() & ((s <= 0) | (s > 120))),
    ("SibSp", "entero >= 0", lambda s: s.notna() & ((s < 0) | (s % 1 != 0))),
    ("Parch", "entero >= 0", lambda s: s.notna() & ((s < 0) | (s % 1 != 0))),
    ("Fare", ">= 0", lambda s: s.notna() & (s < 0)),
    ("Embarked", "C, Q o S", lambda s: s.notna() & ~s.isin(["C", "Q", "S"])),
    ("Cabin", "letra A-G o T seguida de número", lambda s: s.notna() & ~s.astype(str).str.match(r"^[A-GT]\d*(\s[A-GT]?\d*)*$")),
]

SOSPECHOSOS = [
    ("Fare", "Tarifa igual a 0", df["Fare"] == 0, "¿Error de captura o pasaje de cortesía?"),
    ("Age", "Edad menor a 1 año", df["Age"] < 1, "Bebés con edad fraccionaria (válido, pero poco frecuente)"),
    ("Age", "Edad ausente", df["Age"].isna(),""),
]

In [ ]:
def revisar_dominio(df, reglas):
    """reglas: lista de (columna, dominio esperado en texto, función que devuelve la máscara de valores inválidos)."""
    filas = []
    for col, dominio, invalido in reglas:
        s = df[col]
        es_num = pd.api.types.is_numeric_dtype(s) and col in NUMERICAS
        if es_num:
            observado = f"{s.min():,.2f} a {s.max():,.2f}"
        else:
            vals = sorted(s.dropna().astype(str).unique())
            observado = ", ".join(vals[:6]) + (f" … ({len(vals)} en total)" if len(vals) > 6 else "")
        filas.append({"Columna": col, "Dominio esperado": dominio, "Valores observados": observado,
                      "Valores fuera del dominio": int(invalido(s).sum())})
    t = pd.DataFrame(filas).set_index("Columna")
    est = bonito(t, "Dominio de cada columna")
    return aplicar(est, color_alerta, ["Valores fuera del dominio"])

revisar_dominio(df, DOMINIO)

In [ ]:
def valores_sospechosos(df, criterios):
    """Valores que no son inválidos pero conviene revisar. criterios: (columna, criterio, máscara, comentario)."""
    filas = [{"Columna": c, "Criterio": crit, "Filas": int(m.sum()), "Comentario": com}
             for c, crit, m, com in criterios]
    t = pd.DataFrame(filas)   # una misma columna puede aparecer en varias filas
    est = bonito(t, "Valores a revisar").hide(axis="index")
    return aplicar(est, color_alerta, ["Filas"])

valores_sospechosos(df, SOSPECHOSOS)

Ningún valor queda fuera de su dominio: las edades están entre 0.42 y 80, no hay tarifas negativas y los puertos son solo C, Q o S. Aun así, hay tres casos que conviene revisar.

Primero, hay 15 tarifas iguales a 0. Pueden ser pasajes de cortesía o un dato que nunca se registró, y el dataset no permite saber cuál de las dos. Segundo, hay 7 edades menores a un año: son bebés, así que son válidas. Y tercero, las 177 edades ausentes, que habrá que imputar más adelante.

## 2.7 Revisiones adicionales solicitadas

### Filas duplicadas
Las contamos de dos maneras: con todas las columnas y sin los identificadores. Lo hacemos así porque un identificador único hace que ninguna fila parezca repetida, aunque el resto de los datos sea idéntico.

**Ojo:** al quitar los identificadores pueden aparecer coincidencias que no son registros repetidos, sino pasajeros distintos que comparten datos (por ejemplo, personas de una misma familia con el mismo boleto). Por eso conviene revisarlas antes de eliminarlas.

In [ ]:
def filas_duplicadas(df):
    sin_id = df.drop(columns=IDENTIFICADORES)
    t = pd.DataFrame({"Filas": {
        "Duplicadas (todas las columnas)": int(df.duplicated().sum()),
        "Duplicadas (sin considerar identificadores)": int(sin_id.duplicated().sum()),
    }})
    t["% del total"] = (t["Filas"] / len(df) * 100).round(2)
    est = bonito(t, "Filas duplicadas").format({"Filas": "{:,}", "% del total": "{:.2f}%"})
    return aplicar(est, color_alerta, ["Filas"])

filas_duplicadas(df)

Para entender mejor esas coincidencias, miramos de cerca las filas repetidas (sin los identificadores):

In [ ]:
repetidas = df[df.drop(columns=IDENTIFICADORES).duplicated(keep=False)].sort_values(["Ticket", "Age"])
bonito(repetidas[["Ticket", "Pclass", "Sex", "Age", "Fare", "Embarked", "Survived"]].reset_index(drop=True),
       "Un vistazo a las filas repetidas (sin identificadores)").hide(axis="index")

Con todas las columnas no hay ninguna fila duplicada (0), algo esperable porque `PassengerId` y `Name` son únicos. Al quitar los identificadores salen 15 filas repetidas (1.68%), que involucran 25 registros si contamos cada fila junto con sus copias.

Al revisarlas de cerca, las coincidencias vienen en grupos que comparten el mismo número de boleto, la misma tarifa y el mismo puerto. Lo más probable es que sean pasajeros distintos que viajaban juntos, no registros capturados dos veces. Por eso **no las eliminamos**.

### Columnas constantes o casi constantes
Una columna donde casi todas las filas tienen el mismo valor casi no ayuda a distinguir a unos pasajeros de otros. Como criterio, la consideramos casi constante si su valor más frecuente concentra más del 99% de las filas.

In [ ]:
def columnas_constantes(df, umbral=0.99):
    """Una columna es casi constante si su valor más frecuente concentra más del 99% de las filas."""
    t = pd.DataFrame({
        "Valor más frecuente": [str(df[c].value_counts().index[0]) for c in df.columns],
        "% de filas con ese valor": [round(df[c].value_counts(normalize=True).iloc[0] * 100, 2) for c in df.columns],
    }, index=df.columns)
    t["Constante o casi constante"] = np.where(t["% de filas con ese valor"] / 100 > umbral, "Sí", "No")
    t = t.sort_values("% de filas con ese valor", ascending=False)
    est = bonito(t, f"Columnas constantes o casi constantes (umbral: {umbral:.0%})")
    est = est.format({"% de filas con ese valor": "{:.2f}%"})
    return aplicar(est, color_alerta, ["Constante o casi constante"])

columnas_constantes(df)

Ninguna columna es constante ni casi constante. Las más concentradas son `Parch` (76.09% con valor 0), `Embarked` (72.44% en `S`) y `SibSp` (68.24% con 0), pero todas están lejos del 99% que usamos como umbral.

Un detalle: `Embarked` aparece aquí con 72.44%, mientras que en las frecuencias tenía 72.28%. No es un error. Esta tabla calcula el porcentaje sobre los 889 pasajeros que sí tienen puerto, y las frecuencias lo calculan sobre los 891 (contando los 2 ausentes).

### Categorías inconsistentes
Primero revisamos el formato: espacios sobrantes, mayúsculas o guiones que harían distintas a categorías que en realidad son la misma. Además, en el Titanic el título que aparece dentro de `Name` (Mr, Mrs, Miss...) se escribe con variantes equivalentes, así que también lo revisamos.

In [ ]:
def categorias_inconsistentes(df, cols):
    """Busca categorías que serían la misma tras quitar espacios, mayúsculas o guiones."""
    filas = []
    for c in cols:
        s = df[c].dropna().astype(str)
        normal = s.str.strip().str.lower().str.replace(r"[\s_\-]+", " ", regex=True)
        filas.append({"Columna": c, "Categorías distintas": s.nunique(),
                      "Distintas tras normalizar": normal.nunique(),
                      "Valores con espacios sobrantes": int((s != s.str.strip()).sum())})
    t = pd.DataFrame(filas).set_index("Columna")
    t["Inconsistencias"] = np.where((t["Categorías distintas"] != t["Distintas tras normalizar"]) |
                                    (t["Valores con espacios sobrantes"] > 0), "Sí", "No")
    est = bonito(t, "Revisión de formato en las categorías")
    return aplicar(est, color_alerta, ["Inconsistencias"])

categorias_inconsistentes(df, CATEGORICAS)

In [ ]:
# Categorías equivalentes dentro del nombre: el título (Mr, Mrs, Miss, Mlle...) se escribe de varias formas
titulos = df["Name"].str.extract(r",\s*([^\.]+)\.")[0]
equivalencias = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
t = titulos.value_counts().to_frame("Frecuencia")
t["Equivale a"] = [equivalencias.get(x, "—") for x in t.index]
t.index.name = "Título extraído de Name"
est = bonito(t, "Títulos en Name: algunos son variantes del mismo título")
est = est.format({"Frecuencia": "{:,}"})
aplicar(est, lambda v: AMARILLO if v not in ("—",) else "", ["Equivale a"])

El formato de las columnas está limpio: no hay espacios sobrantes ni variantes por mayúsculas, por eso la primera tabla marca "No" en todas.

La inconsistencia está escondida dentro de `Name`. Al extraer el título salen 17 distintos, y algunos son lo mismo escrito de otra forma: `Mlle` y `Ms` equivalen a `Miss` (3 pasajeras) y `Mme` equivale a `Mrs` (1). Además, varios títulos aparecen una sola vez o muy pocas (`Capt`, `Don`, `Lady`, `Sir`, `Jonkheer`, `the Countess`...). `Name` como tal no sirve para modelar, pero el título sí puede ser informativo. Si se usa, conviene unificar las variantes y agrupar los títulos raros en una sola categoría.

### Identificadores únicos
Buscamos columnas con (casi) todos los valores distintos. Un identificador sirve para nombrar a un pasajero, no para describirlo, así que no aporta patrones generales y se deja fuera del modelo.

In [ ]:
def identificadores(df, umbral=95):
    t = pd.DataFrame({
        "Valores únicos": df.nunique(),
        "% de únicos": (df.nunique() / len(df) * 100).round(2),
    })
    t["Único por fila"] = np.where(t["Valores únicos"] == len(df), "Sí", "No")
    t = t[t["% de únicos"] > umbral].sort_values("% de únicos", ascending=False)
    if t.empty:
        print("No hay columnas con más de", umbral, "% de valores únicos.")
        return None
    est = bonito(t, "Posibles identificadores")
    est = est.format({"Valores únicos": "{:,}", "% de únicos": "{:.2f}%"})
    return aplicar(est, color_alerta, ["Único por fila"])

identificadores(df)

`PassengerId` y `Name` son únicos en las 891 filas, así que los excluimos del modelo: identifican a cada persona, pero no describen ningún patrón. `Ticket` no alcanza el umbral del 95% (tiene 76.4%), pero se comporta casi como un identificador para quienes viajaban solos y se repite entre quienes viajaban en grupo. Por eso lo revisamos con más detalle en la sección de fugas.

### Correlaciones elevadas
Calculamos la correlación de Pearson entre las variables numéricas. El umbral `UMBRAL_CORR` define qué consideramos alto. Cuando dos variables están muy correlacionadas, aportan información parecida, es decir, redundante.

In [ ]:
def color_correlacion(v):
    if pd.isna(v):
        return ""
    intensidad = int(abs(v) * 150)
    if v >= 0:
        return f"background-color:rgb(255,{255 - intensidad},{255 - intensidad});color:#000"
    return f"background-color:rgb({255 - intensidad},{255 - intensidad},255);color:#000"

def matriz_correlacion(df, cols):
    corr = df[cols].corr()
    est = bonito(corr, "Correlación de Pearson (rojo: positiva, azul: negativa)").format("{:.2f}")
    return aplicar(est, color_correlacion, None)

def pares_correlacion_alta(df, cols, umbral=UMBRAL_CORR):
    corr = df[cols].corr()
    pares = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(corr.columns)
             for b in corr.columns[i + 1:] if abs(corr.loc[a, b]) >= umbral]
    if not pares:
        print(f"No hay pares de variables con |r| >= {umbral}.")
        return None
    t = pd.DataFrame(pares, columns=["Variable A", "Variable B", "r"]).set_index("Variable A")
    return bonito(t, f"Correlaciones elevadas (|r| >= {umbral})").format({"r": "{:.3f}"})

display(matriz_correlacion(df, COLS_CORR))
altas = pares_correlacion_alta(df, COLS_CORR)
if altas is not None:
    display(altas)

Ningún par llega al umbral de 0.7, así que no hay redundancia fuerte entre estas variables. La relación más notable es la de `Pclass` con `Fare` (−0.55). El signo es negativo porque la primera clase se codifica como 1 y es la que paga más: a menor número de clase, mayor tarifa. También hay relaciones moderadas entre `SibSp` y `Parch` (0.41), lo cual tiene sentido porque las familias suelen tener ambos tipos de parientes, y entre `Pclass` y `Age` (−0.37): los pasajeros de más edad solían ir en mejores clases.

### 2.8 Posibles fugas de información
Hay **fuga** cuando una variable le da al modelo la respuesta, o una pista casi directa, de una forma que no tendríamos al predecir en la vida real. Puede pasar porque la variable se genera después del resultado, porque se deriva del propio objetivo o porque identifica de forma única a cada registro.

La tabla de abajo calcula tres señales automáticas:
- **Identificador:** la columna es única por fila y no sirve para generalizar.
- **Asociación alta (≥ 0.5):** una variable muy ligada al objetivo merece revisión, pero **no es fuga por sí sola**: puede ser un buen predictor que sí conocemos antes del resultado.
- **Ausencia informativa:** que falte el dato se relaciona con el objetivo (una diferencia de 10 puntos porcentuales o más).

Las señales solo avisan. El criterio final lo ponemos nosotros con una pregunta sencilla: ¿tendríamos esa variable disponible en el momento real de decidir?

In [ ]:
def cramers_v(x, y):
    """Asociación entre dos variables categóricas (0 = ninguna, 1 = total)."""
    t = pd.crosstab(x, y)
    n = t.values.sum()
    esperado = np.outer(t.sum(axis=1), t.sum(axis=0)) / n
    chi2 = ((t.values - esperado) ** 2 / esperado).sum()
    k = min(t.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else np.nan

def senales_fuga(df):
    """Para cada variable calcula: asociación con el objetivo y si la ausencia del dato se relaciona con el objetivo."""
    y = (df[OBJETIVO] == POSITIVA).astype(int)
    filas = []
    for c in df.columns:
        if c == OBJETIVO:
            continue
        pct_aus = df[c].isna().mean() * 100
        metrica, asoc = "—", np.nan
        if c in IDENTIFICADORES:
            pass
        elif pd.api.types.is_numeric_dtype(df[c]) and (c in NUMERICAS or df[c].nunique() <= 10):
            asoc, metrica = abs(df[c].corr(y)), "|r|"
        elif df[c].nunique() <= 50:
            asoc, metrica = cramers_v(df[c].fillna("(ausente)"), y), "V de Cramér"
        # Diferencia en la tasa del objetivo entre filas con dato ausente y con dato presente
        dif = np.nan
        if pct_aus >= 5:
            dif = (y[df[c].isna()].mean() - y[df[c].notna()].mean()) * 100
        señales = []
        if c in IDENTIFICADORES:
            señales.append("Identificador: excluir del modelo")
        if not np.isnan(asoc) and asoc >= 0.5:
            señales.append("Asociación alta con el objetivo: revisar")
        if not np.isnan(dif) and abs(dif) >= 10:
            señales.append("La ausencia del dato se relaciona con el objetivo")
        if c not in IDENTIFICADORES and metrica == "—":
            señales.append("Muchas categorías: asociación no calculada")
        filas.append({"Variable": c, "% de ausencia": round(pct_aus, 2), "Métrica": metrica,
                      "Asociación con el objetivo": asoc,
                      "Dif. si falta el dato (pp)": dif,
                      "Señal": "; ".join(señales) if señales else "—"})
    return pd.DataFrame(filas).set_index("Variable")

senales = senales_fuga(df)
est = bonito(senales, "Señales de posible fuga de información")
est = est.format({"% de ausencia": "{:.2f}%", "Asociación con el objetivo": "{:.3f}",
                  "Dif. si falta el dato (pp)": "{:+.1f}"}, na_rep="—")
est = aplicar(est, lambda v: AMARILLO if any(k in v for k in ("Identificador", "Asociación alta", "ausencia")) else "", ["Señal"])
est

`Sex` es la única variable con asociación alta con el objetivo (V de Cramér de 0.543). Es lo esperable: sobrevivió el 74.2% de las mujeres frente al 18.9% de los hombres. Aun así, no es una fuga, porque el sexo de una persona se conoce de antemano. Es un predictor legítimo, simplemente muy fuerte.

Las otras señales son de ausencia informativa. Cuando falta la edad, la tasa de supervivencia es 11.2 puntos menor, y cuando falta la cabina, 36.7 puntos menor. O sea, que el dato falte no es azaroso, y por eso, al imputar, conviene conservar una columna que indique que el dato faltaba. `Ticket` y `Cabin` tienen demasiadas categorías para calcular su asociación, así que los revisamos con la evidencia de abajo.

**Evidencia para el criterio.** Dos columnas merecen una revisión más de cerca: `Cabin` (77% de ausentes) y `Ticket` (el mismo número de boleto lo comparten varios pasajeros).

In [ ]:
# Evidencia: ¿qué dice la presencia de Cabin y el boleto compartido?
y = df["Survived"]
grupos = pd.DataFrame({
    "Pasajeros": [df["Cabin"].notna().sum(), df["Cabin"].isna().sum()],
    "Tasa de supervivencia": [y[df["Cabin"].notna()].mean(), y[df["Cabin"].isna()].mean()],
    "Clase promedio (Pclass)": [df.loc[df["Cabin"].notna(), "Pclass"].mean(), df.loc[df["Cabin"].isna(), "Pclass"].mean()],
}, index=["Con cabina registrada", "Sin cabina registrada"])
display(bonito(grupos, "Cabin: la presencia del dato se asocia con la clase y con sobrevivir")
        .format({"Pasajeros": "{:,}", "Tasa de supervivencia": "{:.1%}", "Clase promedio (Pclass)": "{:.2f}"}))

compartidos = df["Ticket"].duplicated(keep=False)
boletos = pd.DataFrame({"Pasajeros": [int(compartidos.sum()), int((~compartidos).sum())],
                        "Tasa de supervivencia": [y[compartidos].mean(), y[~compartidos].mean()]},
                       index=["Con boleto compartido", "Con boleto único"])
bonito(boletos, "Ticket: si un mismo grupo queda repartido entre entrenamiento y prueba, hay riesgo de fuga") \
    .format({"Pasajeros": "{:,}", "Tasa de supervivencia": "{:.1%}"})

Los pasajeros con cabina registrada sobrevivieron mucho más (66.7%) que los que no la tienen (30.0%), pero buena parte de la diferencia la explica la clase: quienes tienen cabina van en promedio en clase 1.20 (casi todos de primera), frente a 2.64 de quienes no. Es decir, `Cabin` funciona sobre todo como un reflejo de la clase del pasajero, y lo que importa es más si el dato existe que cuál es su contenido. Además, según cómo se haya recolectado esa información, podría incluir datos posteriores al hundimiento. No lo podemos comprobar con el dataset, y es una razón más para usarla con cautela.

Con `Ticket`, quienes compartían boleto con alguien (344 pasajeros) sobrevivieron más (52.0%) que quienes viajaban con boleto único (29.8%). Esto abre un riesgo de fuga por grupo: si al dividir los datos en entrenamiento y prueba se separa a miembros de un mismo grupo, el modelo podría "reconocer" el boleto y adivinar el resultado a partir de los compañeros que ya vio en el entrenamiento. Para evitarlo se puede dividir por grupo de boleto en lugar de por fila, o simplemente no usar `Ticket` como variable.

## 2.9 Producto parcial: hallazgos más importantes
Esta tabla se arma sola a partir de todo lo que calculamos arriba y junta los hallazgos principales en un solo lugar. Sirve como punto de partida para el producto parcial.

In [ ]:
HALLAZGOS_EXTRA = [
    ("Cabin", f"Solo {df['Cabin'].notna().sum()} de {len(df)} pasajeros tienen cabina; su presencia se asocia con primera clase "
              f"(supervivencia {df.loc[df['Cabin'].notna(), 'Survived'].mean():.1%} con cabina vs {df.loc[df['Cabin'].isna(), 'Survived'].mean():.1%} sin ella)."),
    ("Ticket", f"{int(df['Ticket'].duplicated().sum())} filas repiten número de boleto (grupos o familias)."),
    ("Fare", f"{int((df['Fare'] == 0).sum())} tarifas iguales a 0 que conviene revisar."),
]

def hallazgos(df):
    h = []
    aus = (df.isna().mean() * 100).round(2)
    aus = aus[aus > 0].sort_values(ascending=False)
    if len(aus):
        h.append(("Valores ausentes", "; ".join(f"{c}: {v:.2f}%" for c, v in aus.items())))
    h.append(("Filas duplicadas", f"{int(df.duplicated().sum())} (todas las columnas); "
                                  f"{int(df.drop(columns=IDENTIFICADORES).duplicated().sum())} sin considerar identificadores"))
    casi = [c for c in df.columns if df[c].value_counts(normalize=True).iloc[0] > 0.99]
    h.append(("Columnas constantes o casi constantes", ", ".join(casi) if casi else "Ninguna"))
    ids = [c for c in df.columns if df[c].nunique() / len(df) > 0.95]
    h.append(("Identificadores únicos", ", ".join(ids) if ids else "Ninguno"))
    asim = df[NUMERICAS].skew()
    fuertes = asim[asim.abs() > 1].sort_values(key=abs, ascending=False)
    h.append(("Asimetría fuerte (|asimetría| > 1)",
              "; ".join(f"{c}: {v:.2f}" for c, v in fuertes.items()) if len(fuertes) else "Ninguna"))
    corr = df[COLS_CORR].corr()
    pares = [f"{a}–{b}: {corr.loc[a, b]:.3f}" for i, a in enumerate(corr.columns)
             for b in corr.columns[i + 1:] if abs(corr.loc[a, b]) >= UMBRAL_CORR]
    h.append((f"Correlaciones elevadas (|r| >= {UMBRAL_CORR})", "; ".join(pares) if pares else "Ninguna"))
    marcadas = senales[senales["Señal"].str.contains("Identificador|Asociación alta|ausencia", regex=True)]
    if len(marcadas):
        h.append(("Señales de posible fuga", "; ".join(f"{c} ({s})" for c, s in marcadas["Señal"].items())))
    h.extend(HALLAZGOS_EXTRA)
    t = pd.DataFrame(h, columns=["Tema", "Detalle"]).set_index("Tema")
    return bonito(t, "Producto parcial: hallazgos más importantes")

hallazgos(df)

Si hubiera que quedarse con lo más importante para el producto parcial, serían tres cosas: los datos faltantes (`Cabin` y `Age`) y que su ausencia se relaciona con la supervivencia, la fuerte asimetría de `Fare`, `SibSp` y `Parch`, y el riesgo de fuga por grupo en `Ticket`. El resto del dataset está limpio: no hay valores fuera de dominio ni duplicados reales.